In [1]:
import re
import json
import logging
from pathlib import Path
from typing import Dict

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

MISSING_JSON_TEMPLATE = {
    "directed": True,
    "multigraph": True,
    "graph": {
        # will be filled with frame name, e.g. "frame-000090"
        "scene_name": None
    },
    "nodes": [],
    "links": []
}

FRAME_RE = re.compile(r"^frame-(\d{6})\.json$")


def fill_missing_graphs(graphs_root: Path, start_index: int = 0, dry_run: bool = False) -> Dict:
    """
    Пройти по всем папкам сцены в graphs_root и заполнить отсутствующие последовательные frame-######.json
    :param graphs_root: Path к папке, где лежат папки сцен (например .../3RScan/Splited_graphs)
    :param start_index: с какого индекса начинать (по умолчанию 0)
    :param dry_run: если True — только отчёт, без записи файлов
    :return: словарь-отчёт {created: int, skipped_scenes: [..], scenes_checked: int}
    """
    if not graphs_root.exists():
        raise FileNotFoundError(f"{graphs_root} does not exist")

    created = 0
    skipped_scenes = []
    scenes_checked = 0

    for scene_dir in sorted(graphs_root.iterdir()):
        if not scene_dir.is_dir():
            continue
        scenes_checked += 1

        # Найти все json-файлы формата frame-######.json
        json_files = [p.name for p in scene_dir.iterdir() if p.is_file() and p.suffix == ".json" and FRAME_RE.match(p.name)]
        if not json_files:
            logging.info(f"[SKIP] сцена '{scene_dir.name}': не найдено frame-*.json (папка пропущена).")
            skipped_scenes.append((scene_dir.name, "no_frame_jsons"))
            continue

        # извлечь индексы
        indices = sorted({int(FRAME_RE.match(n).group(1)) for n in json_files})
        max_idx = max(indices)
        # начало диапазона — обычно 0, но можно менять стартовый индекс
        start = start_index

        # если в папке есть файлы, которые начинаются не с 0 и ты хочешь начинать с min(indices),
        # заменяй start = min(indices) — но по задаче у тебя начинается с нуля.
        if start > max_idx:
            logging.info(f"[SKIP] сцена '{scene_dir.name}': start_index ({start}) > max_idx ({max_idx}) — пропускаю.")
            skipped_scenes.append((scene_dir.name, "start_gt_max"))
            continue

        # формируем множество существующих индексов для быстрой проверки
        existing = set(indices)

        # найдем отсутствующие индексы в диапазоне start..max_idx
        missing = [i for i in range(start, max_idx + 1) if i not in existing]
        if not missing:
            logging.info(f"[OK] сцена '{scene_dir.name}': все графы {start:06d}-{max_idx:06d} присутствуют.")
            continue

        # выводим информацию и (если не dry_run) создаём пустые json'ы
        logging.warning(f"[MISSING] сцена '{scene_dir.name}': найдено {len(missing)} пропуск(ов). "
                        f"Будут созданы пустые json для: {', '.join(f'frame-{i:06d}.json' for i in missing)}")

        if dry_run:
            # не создаём файлы, просто отчёт
            continue

        for idx in missing:
            name = f"frame-{idx:06d}.json"
            target_path = scene_dir / name
            # формируем content, указываем scene_name как имя файла-фрейма
            content = dict(MISSING_JSON_TEMPLATE)  # поверхностная копия
            content["graph"] = dict(MISSING_JSON_TEMPLATE["graph"])
            content["graph"]["scene_name"] = name
            # Запись атомарно: сначала временный файл, потом replace
            try:
                tmp_path = scene_dir / (name + ".tmp")
                with tmp_path.open("w", encoding="utf-8") as f:
                    json.dump(content, f, indent=2, ensure_ascii=False)
                    f.write("\n")
                tmp_path.replace(target_path)  # atomic on most OS
                created += 1
                logging.info(f"[CREATED] {target_path}")
            except Exception as e:
                logging.exception(f"Failed to write {target_path}: {e}")
                skipped_scenes.append((scene_dir.name, f"write_error:{name}"))

    report = {"created": created, "skipped_scenes": skipped_scenes, "scenes_checked": scenes_checked}
    logging.info(f"Done. created={created}, scenes_checked={scenes_checked}, skipped={len(skipped_scenes)}")
    return report



root = Path("/mnt/external_usb_hdd/6YL/Datasets/3RScan/SceneGraphs")
# dry_run=True — только просмотр, без записи
result = fill_missing_graphs(root, start_index=0, dry_run=False)
print("Report:", result)

INFO: [OK] сцена '05c6ede7-2e69-23b1-8b27-c1cb868f1938': все графы 000000-000137 присутствуют.
INFO: [OK] сцена '095821f7-e2c2-2de1-9568-b9ce59920e29': все графы 000000-000202 присутствуют.
INFO: [OK] сцена '095821f9-e2c2-2de1-9707-8f735cd1c148': все графы 000000-000360 присутствуют.
INFO: [OK] сцена '095821fb-e2c2-2de1-94df-20f2cb423bcb': все графы 000000-000259 присутствуют.
INFO: [OK] сцена '09582205-e2c2-2de1-9475-1cdac7639e60': все графы 000000-000211 присутствуют.
INFO: [OK] сцена '09582207-e2c2-2de1-972c-225d968c2ab4': все графы 000000-000328 присутствуют.
INFO: [OK] сцена '09582209-e2c2-2de1-9610-08baed932919': все графы 000000-000291 присутствуют.
INFO: [OK] сцена '0958220b-e2c2-2de1-96bc-739f09c1e8f8': все графы 000000-000188 присутствуют.
INFO: [OK] сцена '0958220d-e2c2-2de1-9710-c37018da1883': все графы 000000-000191 присутствуют.
INFO: [OK] сцена '09582212-e2c2-2de1-9700-fa44b14fbded': все графы 000000-000156 присутствуют.
INFO: [OK] сцена '09582214-e2c2-2de1-956a-64d8da4b

Report: {'created': 0, 'skipped_scenes': [], 'scenes_checked': 1335}
